### KNN From Scratch
- Objective : Regrssion on sklearn's diabetes data using KNN from scratch

In [12]:
'''
- Goal is to implement regression using KNN
- Implemntation from scratch
- Optimization isnt the ultimate target, clarity is
- Heap is used for handling the nearest neighbors
- We will use diabetes dataset because its lightweight
'''
import heapq
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

In [13]:
X, y = load_diabetes(return_X_y=True)
print(X.shape)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

(442, 10)


In [14]:
'''
- Simplifying k estimation
- Ideal way is to work with KNN iteratively on set of folds compute RMSE and take the k giving least error
'''
k = int(np.sqrt(len(X_train)))
print(k)

18


In [15]:
def EucDist(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

In [16]:
'''
- heapq by default is a min heap.
- To simulate a max heap, we store negative distances. The root of the heap always represents the farthest
- Neighbour currently retained
'''
def fndNghbrs(x_test, X_train, k):
    maxHeap = []

    for trainIndx, x_train in enumerate(X_train):
        distance = EucDist(x_test, x_train)

        if len(maxHeap) < k: #Heap complketion 
            heapq.heappush(maxHeap, (-distance, trainIndx))

        else:
            farthestDist = -maxHeap[0][0] #Convert back to positive distance
            
            if distance < farthestDist: #Replace only if current sample is closer
                heapq.heapreplace(maxHeap, (-distance, trainIndx))

    nghbrIndices = [indx for negDist, indx in maxHeap] #Get neighbor indices
    return nghbrIndices

In [17]:
'''
- For regression tasks, simple averaging is used 
'''
def predict(nghbrIndices, y_train):
    nghbrPts = []

    for indx in nghbrIndices:
        nghbrPts.append(y_train[indx])

    result = np.mean(nghbrPts)

    return result

In [18]:
'''
- RMSE = root(mean(square(diff(y_cap,y_gt))))
'''
def RMSE(y_cap, y_gt):
    return np.sqrt(np.mean((y_cap - y_gt) ** 2))

In [23]:
def KNN(X_test, X_train, y_train, y_test, k):
    y_cap = []
    results = {}

    for tstIndx, xTstPts in enumerate(X_test):

        nghbrIndices = fndNghbrs(xTstPts, X_train, k) #FInd NN

        result = predict(nghbrIndices, y_train) # Regression part

        y_cap.append(result)

        results[tstIndx] = {
            "nghbrIndices": nghbrIndices,
            "y_pred": result,
            "y_gt": y_test[tstIndx],
            "err": abs(result - y_test[tstIndx])
        }

    y_cap = np.array(y_cap)
    score = RMSE(y_cap, y_test)

    return y_cap, results, score

In [24]:
y_cap, results, score = KNN(X_test, X_train, y_train, y_test, k)
print(f"RMSE : {np.round(score, decimals=2)}")

RMSE : 51.42


In [25]:
for i in range(5):
    print(f"Sample {i+1}")
    print(50*"-")
    print(f"Prediction  : {np.round(results[i]['y_pred'], decimals=2)}")
    print(f"Actual      : {np.round(results[i]['y_gt'], decimals=2)}")
    print(f"Difference  : {np.round(results[i]['err'], decimals=2)}")
    print("\n\n")

Sample 1
--------------------------------------------------
Prediction  : 162.11
Actual      : 246.0
Difference  : 83.89



Sample 2
--------------------------------------------------
Prediction  : 189.78
Actual      : 232.0
Difference  : 42.22



Sample 3
--------------------------------------------------
Prediction  : 100.56
Actual      : 187.0
Difference  : 86.44



Sample 4
--------------------------------------------------
Prediction  : 128.78
Actual      : 168.0
Difference  : 39.22



Sample 5
--------------------------------------------------
Prediction  : 100.5
Actual      : 72.0
Difference  : 28.5



